# 14 · Out-of-fold predictions, resumable

13b trains one network and scores a held-out tenth. This trains it
`N_FOLDS` times, each time holding out a different fifth, so that every
labelled product ends up with a prediction from a network that never saw
it. That file is what the second-stage model — the logistic regression
combining this network with the Naive Bayes classifier — is fitted on.

Each fold is 13b's recipe, unchanged, on the rows outside the fold:
validation split, token pruning, vectorizer, label tree, network, early
stopping, temperature calibration. The held-out rows are then scored
through `predict.score`, the path a new product takes.

**Run 13a first.** This reads its cleaned file and its
`run_config.joblib`, and re-cleans nothing. It is not tied to a
blacklist arm: the blacklist is one optional setting below.

## Stopping and starting

On CPU a fold takes hours, so the run is built to be left and come back
to.

- **To stop:** set `TIME_BUDGET_HOURS` and it stops itself after the
  epoch in progress, or simply interrupt the cell or let the cluster
  terminate.
- **To continue:** run the notebook again from the top, with the same
  settings. Finished folds are skipped. The interrupted fold rebuilds its
  matrices (a few minutes) and resumes from its last saved epoch.
- **What a hard stop costs:** whatever came after the last save — at most
  one epoch with `CHECKPOINT_EVERY_MINUTES = 0`.

The run lives in `<outputs>/<RUN_NAME>/`, on the workspace mount, so it
survives the cluster. Changing the settings or the data between sessions
is refused rather than mixed in: folds trained under different settings
do not add up to one set of predictions.

In [ ]:
import os, pathlib, sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "nn_classifier").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ---------------------------------------------------------------------
# Which prepared data to cross-validate: the outputs directory the
# prepare notebook wrote its run_config.joblib into. 13a writes to
# "outputs_codes". Set BEFORE anything reads a config -- without it the
# package loads the shared one and folds the wrong data.
# ---------------------------------------------------------------------
OUTPUT_DIR_NAME = "outputs_codes"

ARM_OUTPUT_DIR = str(ROOT / OUTPUT_DIR_NAME)
os.environ["NN_CLASSIFIER_OUTPUT_DIR"] = ARM_OUTPUT_DIR

if not (pathlib.Path(ARM_OUTPUT_DIR) / "run_config.joblib").exists():
    raise FileNotFoundError(
        f"No run_config.joblib in {ARM_OUTPUT_DIR}. Run the prepare "
        "notebook (13a) first -- this folds the data it writes."
    )
print(f"reads and writes: {ARM_OUTPUT_DIR}")

In [ ]:
import json

import numpy as np
import pandas as pd

import nn_classifier as nnc
from nn_classifier import crossval, data, paths
from nn_classifier.config import load_config

assert str(paths.output_dir()) == ARM_OUTPUT_DIR
print(paths.describe())

## 1 · Settings for the run

The three `FOLD_` settings and `N_FOLDS` decide which products land in
which fold, by a hash of the identifier. **They must match the Naive
Bayes project's**, so that both models hold out the same products
together.

`FOLD_KEY_COL` needs to be in both projects' data. To keep products with
identical descriptions in one fold — stricter, since a network then never
sees a held-out product's exact wording — point it at the original
description column in both projects instead.

In [ ]:
RUN_NAME = "cv_oof"          # the run lives in <outputs>/<RUN_NAME>/

N_FOLDS = 5                  # must match the Naive Bayes project
FOLD_SEED = 0                # must match the Naive Bayes project
FOLD_KEY_COL = "ID"          # must match the Naive Bayes project

# Stop by itself after this many hours, finishing the epoch in progress
# first. None runs until every fold is done.
TIME_BUDGET_HOURS = None

# Least time between saves inside a fold. 0 saves after every epoch,
# which is the safe choice; the time each save takes is printed, and if
# it rivals an epoch, raise this.
CHECKPOINT_EVERY_MINUTES = 0

# Run only these folds, e.g. [0, 1] here and [2, 3, 4] on a second
# cluster at the same time. None runs them all, in order.
ONLY_FOLDS = None

# Items to keep out of training, as in 13b. A single-column file of IDs
# in the data folder, or None.
#   "train"  never trained on, but still scored when their fold is held
#            out -- every product keeps a prediction.
#   "all"    removed before folding; they get no prediction at all.
BLACKLIST_FILE = None
BLACKLIST_SCOPE = "train"

# True discards the run in <outputs>/<RUN_NAME>/ and begins again.
RESTART = False

RUN_DIR = paths.output_dir() / RUN_NAME
print(f"run directory: {RUN_DIR}")

## 2 · Model settings

13b's, as they stand after its overrides, gathered into one cell. **If
you change a setting in 13b, change it here**: the out-of-fold
predictions are only useful if they come from the recipe that trains the
deployed model.

They are fingerprinted when the run starts. Editing them part-way through
a run stops the next session with a list of what changed.

In [ ]:
cfg = load_config()
cfg.validate()          # unpickling skips __post_init__, so check explicitly
levels = list(cfg.data.level_columns)

# --- architecture ---
cfg.nn.hidden_dims = (512, 256)
cfg.nn.dropout = 0.2

# --- optimisation ---
cfg.nn.batch_size = 512
cfg.nn.num_workers = 4
cfg.nn.epochs = 100
cfg.nn.lr = 1e-3
cfg.nn.weight_decay = 1e-5
cfg.nn.label_smoothing = 0.05
cfg.nn.patience = 6
cfg.nn.scheduler_patience = 3
cfg.nn.seed = 42
cfg.nn.level_loss_weights = None
cfg.nn.sparse_grad = True

# --- how the levels combine, and how confidence is calibrated ---
cfg.hierarchy.joint_mode = "independent"
cfg.hierarchy.predict_all_levels = True
cfg.hierarchy.top_k = 3

cfg.calibration.method = "temperature"
cfg.calibration.target_accuracy = 0.95
cfg.calibration.default_threshold = 0.60

# --- vectorization and token pruning ---
cfg.sparse.kind = "count"
cfg.sparse.binary = True
cfg.sparse.ngram_range = (1, 2)
cfg.sparse.unordered_bigrams = True
cfg.sparse.max_features = None
cfg.sparse.min_df = 3
cfg.filters.min_token_count = 5

cfg.validate()

# What the prepared data carries, read from its config rather than
# assumed -- so pointing OUTPUT_DIR_NAME at the wrong directory shows up
# here, not as an unexplained change in accuracy.
print(f"data          : {cfg.data.path}")
print(f"vendor marker : {cfg.cleaning.attach_vendor} "
      f"(column {cfg.data.vendor_col!r})")
print(f"code markers  : {list(getattr(cfg.cleaning, 'marker_columns', {}) or {}) or 'none'}")
print(f"device        : {paths.torch_device(cfg.nn.prefer_gpu)}")
print(cfg.sparse)

## 3 · Data

The cleaned file, as 13a wrote it. The rare-class and rare-path filters
are applied to the whole file once, before folding — the same place 13b
applies them, before its own split.

In [ ]:
df = data.load_data(cfg.data)
print(f"{len(df):,} rows")

for col in {FOLD_KEY_COL, cfg.data.id_col}:
    if col not in df.columns:
        raise KeyError(
            f"{col!r} is not in {cfg.data.path}.\n"
            f"  Columns: {list(df.columns)}"
        )

BLACKLIST = set()
if BLACKLIST_FILE:
    _path = pathlib.Path(paths.data_path(BLACKLIST_FILE))
    if not _path.exists():
        raise FileNotFoundError(
            f"BLACKLIST_FILE is set but there is no file at {_path}. Set it "
            "to None to run without one."
        )
    BLACKLIST = data.load_id_list(str(_path), column_name=cfg.data.id_col)
    if BLACKLIST_SCOPE not in ("train", "all"):
        raise ValueError(f"BLACKLIST_SCOPE={BLACKLIST_SCOPE!r}; expected "
                         "'train' or 'all'")
    if BLACKLIST_SCOPE == "all":
        df = data.drop_ids(df, BLACKLIST, cfg.data.id_col, label="rows")

TRAIN_BLACKLIST = BLACKLIST if (BLACKLIST and BLACKLIST_SCOPE == "train") else None
print(f"blacklist: {BLACKLIST_FILE or 'none'}"
      + (f" ({len(BLACKLIST):,} IDs, scope {BLACKLIST_SCOPE!r})" if BLACKLIST else ""))

## 4 · Run, or continue

**The long cell.** Safe to interrupt, and safe to run again: it picks up
where the last session stopped. The table it returns is the state of the
run when it stopped.

In [ ]:
table = crossval.out_of_fold_predictions(
    cfg, df, RUN_DIR,
    n_folds=N_FOLDS, fold_seed=FOLD_SEED, fold_key_col=FOLD_KEY_COL,
    train_blacklist=TRAIN_BLACKLIST,
    only_folds=ONLY_FOLDS,
    time_budget_hours=TIME_BUDGET_HOURS,
    checkpoint_every_minutes=CHECKPOINT_EVERY_MINUTES,
    restart=RESTART,
)
table

### Where the run stands

Reads only small files, so this cell can also be run from a second
notebook while the one above is still going.

In [ ]:
table = crossval.status(RUN_DIR)
ALL_DONE = bool((table["status"] == "done").all())
print("every fold is done" if ALL_DONE
      else "not finished -- run the notebook again to continue")
table

## 5 · The predictions

Only once every fold is done. Four folds of five would look like a
complete file and be missing a fifth of the catalogue.

The fold accuracies should agree closely with each other and with 13b's
validation accuracy. A fold standing apart is worth understanding before
anything is fitted on top of it.

In [ ]:
if not ALL_DONE:
    print("Skipped: the run is not finished.")
else:
    oof = crossval.collect(RUN_DIR)
    print(f"{len(oof):,} products scored out-of-fold")

    rows = []
    for k, part in list(oof.groupby("fold")) + [("all", oof)]:
        row = {"fold": str(k), "rows": len(part)}
        for c in levels:
            row[f"{c} accuracy"] = float(
                (part[f"Predicted {c}"] == part[f"Actual {c}"]).mean())
        rows.append(row)
    by_fold = pd.DataFrame(rows)
by_fold if ALL_DONE else None

In [ ]:
# Does the confidence mean what it says, on rows no network saw? The
# deepest level's own calibrated confidence, in bands. `gap` near zero
# is a calibrated network; consistently negative is over-confidence.
if ALL_DONE:
    deepest = levels[-1]
    conf = oof[f"{deepest} Confidence"].astype(float)
    right = (oof[f"Predicted {deepest}"] == oof[f"Actual {deepest}"])
    bands = pd.cut(conf, [0, .5, .6, .7, .8, .9, .95, .99, 1.0], include_lowest=True)
    reliability = (pd.DataFrame({"band": bands, "confidence": conf, "correct": right})
                   .groupby("band", observed=False)
                   .agg(rows=("correct", "size"), mean_confidence=("confidence", "mean"),
                        accuracy=("correct", "mean")))
    reliability["gap"] = reliability["accuracy"] - reliability["mean_confidence"]
    reliability.index = reliability.index.astype(str)
reliability.reset_index() if ALL_DONE else None

## 6 · Save

One file, one row per product. The sidecar records the fold settings; the
Naive Bayes project writes the same, and the two are compared before the
predictions are joined.

| column | meaning |
|---|---|
| `fold` | the fold that held the row out |
| `Actual <level>` | the true label |
| `Predicted <level>` | the assigned path |
| `<level> Confidence` | that level's calibrated probability for the class predicted — **the one to stack on** |
| `Confidence` | probability of the whole path; the worst-calibrated column here |
| `<level> Prefix Probability` | mass over every path sharing the prefix |
| `Path Evidence` | the joint score un-normalised |
| `n_tokens`, `n_known_tokens`, `known_share` | how much of the description that fold's network could read |

In [ ]:
if not ALL_DONE:
    print("Nothing saved: the run is not finished.")
else:
    out = paths.output_path("oof_predictions_nn.csv")
    oof.to_csv(out, index=False)

    run = json.loads((RUN_DIR / "run.json").read_text(encoding="utf-8"))
    meta = {
        "model": "neural_network",
        "n_rows": int(len(oof)),
        "n_folds": run["n_folds"],
        "fold_seed": run["fold_seed"],
        "fold_key_col": run["fold_key_col"],
        "fold_method": run["fold_method"],
        "id_col": run["id_col"],
        "level_columns": run["level_columns"],
        "joint_mode": cfg.hierarchy.joint_mode,
        "calibration": cfg.calibration.method,
        "train_blacklist": run["train_blacklist"],
        "folds": table.to_dict(orient="records"),
    }
    meta_path = paths.output_path("oof_predictions_nn.json")
    meta_path.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")

    print(f"wrote {len(oof):,} rows to {out}")
    print(f"wrote {meta_path}")
    print(f"columns: {list(oof.columns)}")

## Settings that change what comes back

| | effect |
|---|---|
| `crossval.out_of_fold_predictions(..., prune_held_out=True)` | scores the held-out text with training's rare tokens removed first — what 13b's *validation* rows see. The default leaves the text as it is, which is what `predict.score` does to a new product, and is the one to stack on |
| `model_factory=` | a function `(input_dim, n_classes_per_level, cfg) -> model` to fold a different network. The default builds 13b's `MultiHeadSparseInputMLP` |
| `keep_checkpoints=True` | keeps each fold's final training state instead of deleting it once the fold's predictions are written |
| `checkpoint_max_file_mb=` | the size each checkpoint file is split at. 400 stays under the workspace mount's per-file cap |

Each fold's training curve is in `<outputs>/<RUN_NAME>/fold_<k>/history.csv`.